# 독립 푸드코트 시뮬레이션 · PPO / 평가함수 샘플링

기존 CEM 실험과 별개입니다. 각 평가함수로 모델을 학습한 뒤 같은 함수와 같은 외부 조건으로 AI 개입·미개입을 비교합니다.

기본값은 **작은 실행 검증용 파일럿**입니다. 대규모 학습은 아래 설정을 바꾼 뒤 실행하세요. 결과는 Drive의 별도 폴더에 저장합니다. Colab 실행 종료 후에는 이 노트북에서 같은 설정으로 다시 실행해야 합니다. 자동 재접속 기능은 없습니다.

공개 대학 메뉴 출처는 저장소 `src/kiosk_adaptive/data/sources.json`을 확인하세요. 고객 만족·레시피·조리시간·알레르기는 합성 가정이며 실제 만족도나 실제 알레르기 안전을 입증하지 않습니다.


In [ ]:
# 1. 검증된 GitHub 소스 버전 가져오기 및 설치
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/phj9187/foodcourt-adaptive-sim.git'
SOURCE_COMMIT = '8bc5f3636bee5577e18ffdc68a8331952b5478cd'
WORK = Path('/content/foodcourt-adaptive-sim')
if not WORK.exists():
    subprocess.run(['git','clone',REPOSITORY,str(WORK)],check=True)
else:
    origin = subprocess.check_output(['git','-C',str(WORK),'remote','get-url','origin'],text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError('같은 경로에 다른 프로젝트가 있습니다. 새 런타임을 사용하세요.')
    subprocess.run(['git','-C',str(WORK),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','--detach',SOURCE_COMMIT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(WORK)],check=True)
print('소스 버전:',subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# 2. 핵심 테스트 (모델 학습 실험과는 별개)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=WORK,check=True)


In [ ]:
# 3. Google Drive 연결 · 실험 선택
from google.colab import drive
from datetime import datetime, timezone
import re

drive.mount('/content/drive')
CONFIG = 'config/adaptive_pilot.yaml'  # 대규모 예시: config/adaptive_colab.yaml
EXPERIMENT = 'adaptive_pilot_20261105_001'  # 설정/소스가 달라지면 새 ID를 쓰세요.
if not re.fullmatch(r'[A-Za-z0-9_-]+',EXPERIMENT):
    raise ValueError('실험 ID에는 영문, 숫자, 밑줄, 하이픈만 사용하세요.')
OUTPUT = Path('/content/drive/MyDrive/foodcourt_adaptive_sim')/EXPERIMENT
OUTPUT.mkdir(parents=True,exist_ok=True)
from kiosk_adaptive.study import atomic_json
launch = dict(source_commit=SOURCE_COMMIT,config=CONFIG,experiment=EXPERIMENT)
launch_path = OUTPUT/'launch.json'
if launch_path.exists() and json.loads(launch_path.read_text()) != launch:
    raise RuntimeError('이 폴더는 다른 소스/설정으로 생성됐습니다. 새 실험 ID를 지정하세요.')
atomic_json(launch_path,launch)
print('저장 위치:',OUTPUT)
print((WORK/CONFIG).read_text())


In [ ]:
# 4. 학습 · 쌍대 비교 실행 / 동일 폴더로 재실행하면 이어서 진행
# 운영기간 하나를 마치고 PPO 갱신할 때마다 체크포인트와 직전 사본 저장.
# 중간에 끊기면 아직 완료하지 못한 운영기간만 다시 계산합니다.
import signal
log_path = OUTPUT/'run.log'
status_path = OUTPUT/'run_status.json'
def status(value,**extra):
    atomic_json(status_path,dict(status=value,updated_utc=datetime.now(timezone.utc).isoformat(),**extra))
command=[sys.executable,'-u','-m','kiosk_adaptive.study','--config',str(WORK/CONFIG),'--output',str(OUTPUT)]
status('running',command=command)
process=None
try:
    with log_path.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');log.write(line)
        code=process.wait()
    status('complete' if code==0 else 'failed',exit_code=code)
    if code:raise RuntimeError(f'실행 실패: {code}. run.log를 확인하세요.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try:process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill();process.wait()
    status('interrupted')
    print('중단됨. 같은 설정으로 다시 실행하면 마지막 저장 지점부터 이어갑니다.')
except Exception as exc:
    status('failed',error=str(exc))
    raise


In [ ]:
# 5. 결과 확인 — 함수별 자기 기준의 AI 도입 효과
import csv
path=OUTPUT/'comparison.csv'
if path.exists():
    with path.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    for row in rows:
        print(row['id'], '반복',row['replica'],row['scenario'],
              '전체점수 변화',round(float(row['mean_delta']),6),
              '고객점수 변화',round(float(row['mean_customer_delta']),6),
              '이익 변화(원)',round(float(row['mean_profit_delta_won']),1))
else:
    print('아직 전체 실험이 끝나지 않았습니다. 모델별 progress.json과 summary.json을 확인하세요.')
